In [ ]:
 !nvidia-smi

In [ ]:
!pip install -U ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, os, glob, yaml

zip_name = "/content/drive/MyDrive/plates_dataset.zip"

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall("/content/dataset")

print(os.listdir("/content/dataset"))

In [ ]:
yaml_files = glob.glob("/content/dataset/**/data.yaml", recursive=True)
print("Found data.yaml at:", yaml_files)
DATA_YAML = yaml_files[0]

# make "path" point to where the dataset really is
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = os.path.dirname(DATA_YAML)
with open(DATA_YAML, "w") as f:
    yaml.dump(cfg, f)

print(cfg)

In [ ]:
for split in ["train", "val", "test"]:
    imgs = glob.glob(f"/content/dataset/**/images/{split}/*", recursive=True)
    lbls = glob.glob(f"/content/dataset/**/labels/{split}/*.txt", recursive=True)
    print(split, "-> images:", len(imgs), "| labels:", len(lbls))

In [ ]:
import gc, torch, pandas as pd
from ultralytics import YOLO

PROJECT = "/content/drive/MyDrive/yolo_runs"   # saved on Drive
EPOCHS = 50          # same for every run
rows = []

def run_experiment(weights, imgsz, batch, seed=0):
    name = f"{weights.replace('.pt','')}_img{imgsz}_b{batch}_s{seed}"
    try:
        model = YOLO(weights)
        model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=imgsz, batch=batch,
                    seed=seed, project=PROJECT, name=name, exist_ok=True)
        best_path = str(model.trainer.best)

        best = YOLO(best_path)
        m = best.val(data=DATA_YAML, split="test", imgsz=imgsz, batch=batch,
                     project=PROJECT, name=name + "_test", exist_ok=True)

        rows.append({
            "model": weights, "imgsz": imgsz, "batch": batch, "seed": seed,
            "mAP50": m.box.map50, "mAP50-95": m.box.map,
            "precision": m.box.mp, "recall": m.box.mr,
            "inference_ms": m.speed["inference"],
            "params_M": sum(p.numel() for p in best.model.parameters()) / 1e6,
            "weights_MB": os.path.getsize(best_path) / 1e6,
        })
    except torch.cuda.OutOfMemoryError:
        print(f"OUT OF MEMORY: {name} (lower batch or imgsz)")
        rows.append({"model": weights, "imgsz": imgsz, "batch": batch, "seed": seed})
    finally:
        gc.collect(); torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(f"{PROJECT}/results.csv", index=False)
    return df

In [ ]:
EPOCHS = 3
df = run_experiment("yolo26n.pt", imgsz=640, batch=16)
df

In [ ]:
rows.clear()
EPOCHS = 50

In [ ]:
for w in ["yolo26n.pt", "yolo26s.pt"]:
    df = run_experiment(w, imgsz=640, batch=16)
df

In [ ]:
settings = [(960, 16), (960, 8), (1280, 8), (640, 32)]   # (imgsz, batch)

for w in ["yolo26n.pt", "yolo26s.pt"]:      # or only the one that won in Experiment A
    for imgsz, batch in settings:
        df = run_experiment(w, imgsz=imgsz, batch=batch)

df.sort_values("mAP50-95", ascending=False)